제 1유형

1. 다음의 데이터는 대륙별 국가의 맥주소비량을 조사한 것이다.

1.1. 평균 맥주소비량이 가장 많은 대륙을 구하시오.

1.2. 이전 문제에서 구한 대륙에서 5번째로 맥주소비량이 많은 나라를 구하시오.

1.3. 이전 문제에서 구한 나라의 평균 맥주소비량을 구하시오.




2. 다음의 데이터는 국가별로 방문객 유형을 조사한 것이다.

2.1. 관광객비율이 두 번째로 높은 나라의 '관광' 수를 구하시오.
(관광객비율 = 관광/합계(소수점 넷째 자리에서 반올림))
(합계: 관광 + 사무 + 공무 + 유학 + 기타)

2.2. 관광 수가 두 번째로 높은 나라의 '공무'수의 평균을 구하시오.(소수점 첫째 자리에서 반올림)

2.3. 이전에 구한 관광 수와 공무 수의 합계를 구하시오.



3. CO(GT), NMHC(GT) 칼럼에 대해서 Min-Max 스케일러를 실행하고, 스케일링 된 CO(GT), NMHC(GT) 칼럼의 표준편차를 구하시오.(소수점 셋째 자리에서 반올림)

In [6]:
import pandas as pd
df = pd.read_csv('8_1_1.csv')
#print(df.head())

q1 = df.groupby('대륙')['맥주소비량'].mean().sort_values(ascending=False)
#print(q1.idxmax())
#답: SA

df_1 = df[df['대륙']=='SA']
q2 = df_1.groupby('국가')['맥주소비량'].sum().sort_values(ascending=False)
target_country = q2.index[4]
print(target_country)
#print(q2)
#답: Venezuela

result3 = df[df['국가']=='Venezuela']['맥주소비량'].mean()
print(result3)
#답: 253.0176


Venezuela
253.0176


In [2]:
import pandas as pd
df = pd.read_csv('8_1_2.csv')
#print(df.head())

df['합계'] = df['관광'] + df['사무'] + df['공무'] + df['유학'] + df['기타']
df['관광객비율'] = df['관광'] / df['합계']

df_2 = df.sort_values(by='관광객비율', ascending=False)
#print(df_2.head(3))
#답: 7831

df_c = df[['국가', '관광']].sort_values(by='관광', ascending=False)
#print(df_c)
# 이스라엘
result2 = df[df['국가']=='이스라엘'][['공무']].mean().sort_values(ascending=False)
#print(round(result2))
#답: 494

#2.3. 이전에 구한 관광 수와 공무 수의 합계를 구하시오.
df['관광 + 공무'] = df['관광'] + df['공무']
result3 = df[df['국가']=='이스라엘'][['관광 + 공무']].sum()
#print(result3)
#답: 31796



In [28]:
# 3. CO(GT), NMHC(GT) 칼럼에 대해서 Min-Max 스케일러를 실행하고, 스케일링 된 CO(GT), NMHC(GT) 칼럼의 표준편차를 구하시오.(소수점 셋째 자리에서 반올림)
import pandas as pd
df = pd.read_csv('8_1_3.csv')
#print(df.head())

from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
target_cols = ['CO(GT)', 'NMHC(GT)']
df[target_cols] = scaler.fit_transform(df[target_cols])

std_co = df['CO(GT)'].std()
std_nmhc = df['NMHC(GT)'].std()

print(round(std_co, 2))
print(round(std_nmhc, 2))

0.37
0.15


제 2유형

훈련 데이터로 학습한 모델을 테스트 데이터에 적용하여 예측한 결과를 제출하시오(Target: count)

% 제출 형식은 ID, pred 두 칼럼만 존재해야 한다.(평가 지표: MAE)

In [30]:
import pandas as pd
train = pd.read_csv('8_2_train.csv')
test = pd.read_csv('8_2_test.csv')

#print(train.head())
# 범주형: holiday, workingday, weather
# 제거: ID, (count)
#print(test.info())

X = train.drop(['ID', 'count'], axis=1)
y = train['count']
X_submit = test.drop(['ID', 'count'], axis=1)

combined = pd.concat([X, X_submit])
cols = (['holiday', 'workingday', 'weather'])
combined_encoded = pd.get_dummies(combined, columns=cols)

X = combined_encoded.iloc[:len(X)]
X_submit = combined_encoded.iloc[len(X):]

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestRegressor()
model.fit(X_train, y_train)
pred = model.predict(X_val)

from sklearn.metrics import mean_absolute_error, r2_score
score_mae = mean_absolute_error(y_val, pred)
score_r2 = r2_score(y_val, pred)
print(score_mae)
print(score_r2)

model.fit(X, y)
pred_final = model.predict(X_submit)

result = pd.DataFrame({
    'ID': test['ID'],
    'pred': pred_final
})
#result.to_csv('result.csv', index=False)
#print(pd.read_csv('result.csv'))
print(result)

122.76623088972433
0.29449858725121225
        ID        pred
0     4775   79.266667
1    10539  315.500000
2     8229  121.230000
3     8677  268.339762
4     5071  151.080000
..     ...         ...
161   2573  434.090000
162   3329  532.520000
163   2373  386.640000
164    959  149.370000
165   8589   86.690000

[166 rows x 2 columns]


제 3유형

<3-1: 새 소프트웨어 효과 검증>

1.1. 도입 전과 도입 후의 업무처리 시간의 평균과 표준편차를 구하시오.(소수점 둘째 자리까지 반올림)

1.2. 도입 전후의 업무처리 시간 차이가 유의미한지 부호 순위 검정을 실시하고, 검정 통계량을 계산하시오.(소수점 둘째 자리까지 반올림)

1.3. p-value를 바탕으로 유의수준 5%에서 귀무가설의 기각 채택 여부를 결정하시오.(소수점 셋째 자리까지 반올림)




<3-2: 생산성 영향요인 분석>

2.1. 훈련 데이터를 기준으로 생산성 점수(productivity)를 종속변수로 하고 근무 시간, 연력, 그리고 경력을 독립변수로 하는 다중회귀 분석을 수행한 후, 회귀계수가 가장 높은 변수를 구하시오.(다중회귀모형 적합 시 절편 포함)

2.2. 유의수준 5%하에서 각 독립변수가 생산성에 미치는 영향이 통계적으로 유의미한 지 판단하고, 유의미한 변수 개수를 구하시오.(p-value는 소수점 넷째 자리까지 반올림)

2.3. 테스트 데이터로 모델의 성능을 평가하시오.(R2 산출)



In [36]:
import pandas as pd
df = pd.read_csv('8_3_1.csv')
#print(df.head())
mean_before = df['before'].mean()
mean_after = df['after'].mean()

std_before = df['before'].std()
std_after = df['after'].std()

print(round(mean_before, 2))
print(round(mean_after, 2))
print(round(std_before, 2))
print(round(std_after, 2))

from scipy.stats import wilcoxon
before = df['before']
after = df['after']
stat, p_val = wilcoxon(before, after)
print(round(stat, 2))
print(round(p_val, 3))
# 0.0: 기각
# 만약 문제에서 "도입 후 시간이 감소했는지 검정하시오"라고 했다면 alternative='greater' (before > after) 옵션을 써야 합니다. (순서 주의!)

8.21
7.23
1.71
1.96
72.0
0.0


In [ ]:
# <3-2: 생산성 영향요인 분석>
#2.1. 훈련 데이터를 기준으로 생산성 점수(productivity)를 종속변수로 하고 근무 시간, 연력, 그리고 경력을 독립변수로 하는 다중회귀 분석을 수행한 후, 회귀계수가 가장 높은 변수를 구하시오.(다중회귀모형 적합 시 절편 포함)

#2.2. 유의수준 5%하에서 각 독립변수가 생산성에 미치는 영향이 통계적으로 유의미한 지 판단하고, 유의미한 변수 개수를 구하시오.(p-value는 소수점 넷째 자리까지 반올림)

#2.3. 테스트 데이터로 모델의 성능을 평가하시오.(R2 산출)

import pandas as pd
train = pd.read_csv('8_3_2_train.csv')
test = pd.read_csv('8_3_2_test.csv')
#print(train.head())

from statsmodels.formula.api import ols
model = ols('productivity ~ hours + age + experience', data=train).fit()
print(model.summary())
# 답: hours
p_val = model.pvalues
print(round(p_val, 4))
# 답: 3개

from sklearn.metrics import r2_score
pred = model.predict(test)
score = r2_score(test['productivity'], pred)
print(round(score, 3))
# 답: 0.804

                            OLS Regression Results                            
Dep. Variable:           productivity   R-squared:                       0.732
Model:                            OLS   Adj. R-squared:                  0.721
Method:                 Least Squares   F-statistic:                     69.03
Date:                Mon, 24 Nov 2025   Prob (F-statistic):           1.20e-21
Time:                        16:31:30   Log-Likelihood:                -297.10
No. Observations:                  80   AIC:                             602.2
Df Residuals:                      76   BIC:                             611.7
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     67.2310     11.097      6.059      0.0